[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bethavila/dncp-riesgo-contrataciones/blob/main/notebooks/06_spark_distribuido.ipynb)

# 6. Procesamiento distribuido con Apache Spark
**Proyecto III Big Data — Maestría en Ciencia de Datos (UCOM)** · Bethania Avila, Arturo Fernández, Gustavo Bogado

El pipeline principal usa DuckDB (el motor más rápido en una sola máquina). Este notebook calcula los indicadores clave con **Spark SQL sobre los mismos Parquet** y verifica que los resultados coinciden: el mismo código corre en `local[*]` (Colab) o en un clúster (Dataproc, EMR, Databricks) leyendo desde HDFS/S3/GCS. **Ejecutar en Google Colab** (Spark requiere Java; Colab lo trae instalado).

In [ ]:
# Configuración del entorno (Google Colab o local)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/bethavila/dncp-riesgo-contrataciones.git"  # <- URL del repositorio en GitHub
USAR_DRIVE = False  # True: guarda el data lake en Google Drive y no repite descargas entre sesiones

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/dncp-riesgo-contrataciones")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(RAIZ / "requirements-colab.txt")], check=True)
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["DNCP_DATA_DIR"] = "/content/drive/MyDrive/dncp_data_lake"
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dncp_riesgo import graficos as gr
from dncp_riesgo.pipeline import asegurar_datos, preparar

cfg, lake = preparar()
gr.estilo()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| años:", cfg["fuentes"]["ocp_registry"]["anios"])

In [ ]:
asegurar_datos(lake, cfg, hasta="gold")
try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
from dncp_riesgo.procesamiento.spark_jobs import (crear_sesion, registrar_tablas, indicadores_spark,
                                                  concentracion_entidades_spark, precios_referencia_spark)
spark = crear_sesion()
tablas = registrar_tablas(spark, lake)
print("Spark", spark.version, "| procesos vigentes:", tablas["procesos_v"].count())

In [ ]:
import time
t = time.perf_counter()
ind_spark = indicadores_spark(spark, cfg).cache()
resumen_spark = ind_spark.groupBy("anio").agg({"ocid": "count", "oferente_unico": "avg"}).toPandas().sort_values("anio")
print(f"Spark: {time.perf_counter() - t:.1f} s")
resumen_duck = lake.leer_gold("indicadores").groupby("anio").agg(procesos=("ocid", "count"), oferente_unico=("oferente_unico", "mean"))
comparacion = resumen_spark.set_index("anio").rename(columns={"count(ocid)": "procesos_spark", "avg(oferente_unico)": "oferente_unico_spark"}).join(resumen_duck)
comparacion["coinciden"] = (comparacion.procesos_spark == comparacion.procesos) & np.isclose(comparacion.oferente_unico_spark, comparacion.oferente_unico)
comparacion

In [ ]:
concentracion_entidades_spark(spark, ind_spark).limit(10).toPandas()

In [ ]:
t = time.perf_counter(); n = precios_referencia_spark(spark).count()
print(f"Precio de referencia de {n:,} ítems del catálogo calculado sobre millones de filas en {time.perf_counter() - t:.1f} s".replace(",", "."))

In [ ]:
from dncp_riesgo.procesamiento.benchmark import benchmark_motores
benchmark_motores(lake, spark)

In [ ]:
destino = lake.ruta("gold", "spark_indicadores").as_posix()
ind_spark.write.mode("overwrite").partitionBy("anio").parquet(destino)
print("Escrito en", destino)
spark.stop()